# Chapter 5 - Lab 2: <font color='blue'>The Full Deep Search Pipeline</font>

**<font color='purple'>Goal</font>**:
In this lab you will wire the four Deep Search Agent components together into a single autonomous pipeline:

**Plan → Execute → Validate → (Replan?) → Synthesize**

You will see how:

* The **planner** produces a research DAG (built in Lab 1).
* The **researcher** walks the DAG, running independent tasks in parallel with `asyncio.gather`.
* The **validator** uses deterministic Python checks — not an LLM — to verify findings.
* The **synthesizer** produces a structured `ResearchReport` using PydanticAI's structured output.

To keep this lab runnable without external API keys, we use **stub tools** that return canned data for NVDA, AMD, and INTC. The real-API versions (Financial Datasets, SEC EDGAR, Tavily) live in the chapter's source tree.

**<font color='purple'>Tech stack</font>**:

* **PydanticAI** (`pydantic-ai`) — for planner and synthesizer with structured output.
* **OpenAI** `gpt-4o` — planning + synthesis; `gpt-4o-mini` is fine for routing.
* **Python `asyncio`** — parallel task execution.
* **Stub tools** — embedded in this notebook so the pipeline runs end-to-end.

## 1. Install packages

In [ ]:
%pip install -q pydantic-ai pydantic python-dotenv

## 2. Set up the OpenAI API key

Add `OPENAI_API_KEY` in Colab's *key* menu, or set the environment variable locally.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 5 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%205/common.py',
        'common.py',
    )

from common import (
    ResearchPlan, SubTask, TaskStatus,
    CompanyMetrics, ValidationResult, ResearchReport,
    PLANNING_MODEL, SYNTHESIS_MODEL, MAX_REPLAN_ATTEMPTS,
    extract_ticker, format_plan, format_validation,
)

print('Shared models loaded.')

## 4. Define stub tools

In production these tools call external APIs (Financial Datasets, SEC EDGAR, Tavily). For reproducibility we stub them with canned data for three tickers. **Notice each stub is async** — we want the same call signature as the real tools so swapping them in later is trivial.

In [ ]:
STUB_METRICS = {
    'NVDA': CompanyMetrics(
        ticker='NVDA', company_name='NVIDIA',
        revenue=60_922, revenue_growth=125.8,
        eps=12.96, pe_ratio=72.4,
        gross_margin=72.7, operating_margin=54.1,
        market_cap=2891.0, period='FY2024',
    ),
    'AMD': CompanyMetrics(
        ticker='AMD', company_name='Advanced Micro Devices',
        revenue=22_680, revenue_growth=-4.4,
        eps=0.53, pe_ratio=233.0,
        gross_margin=46.1, operating_margin=1.8,
        market_cap=190.0, period='FY2023',
    ),
    'INTC': CompanyMetrics(
        ticker='INTC', company_name='Intel',
        revenue=54_228, revenue_growth=-13.7,
        eps=0.40, pe_ratio=109.0,
        gross_margin=40.0, operating_margin=-1.8,
        market_cap=130.0, period='FY2023',
    ),
}


async def get_financial_metrics(ticker: str) -> CompanyMetrics:
    if ticker not in STUB_METRICS:
        raise ValueError(f'No stub data for {ticker}')
    return STUB_METRICS[ticker]


async def get_risk_factors(ticker: str) -> str:
    return f'[Risk factors for {ticker} — stub]\n- AI demand cyclicality\n- Geopolitical risk\n- Manufacturing concentration'


async def search_financial_news(query: str) -> str:
    return f'[News search for: {query!r} — stub]\nRecent reports indicate continued AI infrastructure demand.'

## 5. The planner (from Lab 1)

Re-create the planner agent — same definition as Lab 1. In a real project this would import from `planner.py`.

In [ ]:
from pydantic_ai import Agent

PLANNER_SYSTEM_PROMPT = (
    'You are a financial research planner. Given a research question, '
    'decompose it into 5-10 concrete sub-tasks with explicit dependencies. '
    'Available data_sources: financial_api, sec_filing, web_search. '
    'Tasks with no dependencies execute in parallel; final synthesis task depends on all data tasks.'
)

planner_agent = Agent(
    model=PLANNING_MODEL,
    system_prompt=PLANNER_SYSTEM_PROMPT,
    output_type=ResearchPlan,
    retries=2,
)

## 6. The researcher — parallel DAG execution

The researcher walks the plan's DAG. In each iteration we collect *ready* tasks (those whose dependencies are all complete) and run them concurrently with `asyncio.gather`. This is the classic topological wave-front execution.

In [ ]:
analyst_agent = Agent(
    model=SYNTHESIS_MODEL,
    system_prompt=(
        'You are a senior financial analyst. Analyze the provided data '
        'and produce a clear, factual response. Use specific numbers. '
        'Do not fabricate data — if missing, state that explicitly.'
    ),
    retries=1,
)

In [ ]:
import asyncio

async def _route_and_execute(task: SubTask, prior: dict[int, str]) -> str:
    sources = task.data_sources

    # Direct tool routes
    if 'financial_api' in sources and not task.dependencies:
        ticker = extract_ticker(task.description)
        if ticker:
            metrics = await get_financial_metrics(ticker)
            return metrics.model_dump_json(indent=2)
    if 'sec_filing' in sources and 'financial_api' not in sources:
        ticker = extract_ticker(task.description)
        if ticker:
            return await get_risk_factors(ticker)
    if 'web_search' in sources and len(sources) == 1:
        return await search_financial_news(task.description)

    # Otherwise use the analyst agent on the dependency context
    context = '\n\n---\n\n'.join(prior[d] for d in task.dependencies if d in prior)
    prompt = f'Task: {task.description}\n\nAvailable data:\n{context or "No prior data."}'
    result = await analyst_agent.run(prompt)
    return result.output

In [ ]:
async def execute_plan(plan: ResearchPlan) -> dict[int, str]:
    results: dict[int, str] = {}
    tasks_by_id = {t.id: t for t in plan.sub_tasks}

    while any(t.status == TaskStatus.PENDING for t in plan.sub_tasks):
        ready = [
            t for t in plan.sub_tasks
            if t.status == TaskStatus.PENDING
            and all(tasks_by_id[d].status == TaskStatus.COMPLETED for d in t.dependencies)
        ]
        if not ready:
            for t in (x for x in plan.sub_tasks if x.status == TaskStatus.PENDING):
                t.status = TaskStatus.FAILED
                t.result = 'Unresolvable dependencies'
                results[t.id] = t.result
            break

        async def _execute_one(task: SubTask):
            task.status = TaskStatus.IN_PROGRESS
            try:
                r = await _route_and_execute(task, results)
                task.status = TaskStatus.COMPLETED
                task.result = r
                return (task.id, r)
            except Exception as e:
                task.status = TaskStatus.FAILED
                msg = f'[FAILED] {e}'
                task.result = msg
                return (task.id, msg)

        for task_id, r in await asyncio.gather(*[_execute_one(t) for t in ready]):
            results[task_id] = r
            print(f'  [{tasks_by_id[task_id].status.value}] task {task_id}')

    return results

## 7. The validator (deterministic checks)

Validation is **deterministic Python**, not an LLM call. This is the key insight from the AI Alliance Deep Research paper: code-based checks are faster, cheaper, and far more reliable than asking a model 'is this correct?'. We verify task completion, ticker coverage, and the math of the metrics themselves.

In [ ]:
def validate_research(plan: ResearchPlan, results: dict[int, str], required_tickers: list[str]) -> ValidationResult:
    errors, warnings, gaps = [], [], []

    failed = [t for t in plan.sub_tasks if t.status == TaskStatus.FAILED]
    for t in failed:
        gaps.append(f'Task {t.id} failed: {t.description}')

    found = set()
    for r in results.values():
        for tk in required_tickers:
            if tk.upper() in r.upper():
                found.add(tk.upper())
    for missing in (set(t.upper() for t in required_tickers) - found):
        gaps.append(f'Missing data for required ticker: {missing}')

    for r in results.values():
        try:
            m = CompanyMetrics.model_validate_json(r)
            if m.gross_margin > 5 and m.operating_margin > m.gross_margin + 1:
                errors.append(
                    f'{m.ticker}: operating margin ({m.operating_margin}%) '
                    f'exceeds gross margin ({m.gross_margin}%) — impossible'
                )
        except Exception:
            pass  # not all results are metrics

    return ValidationResult(
        is_valid=(not errors and not gaps),
        errors=errors, warnings=warnings, gaps=gaps,
    )

## 8. The synthesizer (structured report)

The synthesizer's job is narrative quality — combining quantitative data and qualitative analysis into a coherent memo. We use PydanticAI's structured output again, this time targeting `ResearchReport`. The model returns a typed object with title, executive summary, metrics, risks, conclusion, and a confidence score.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = (
    'You are a senior equity research analyst. Produce a structured investment memo '
    'with: title, 3-5 sentence executive summary, the company metrics provided, '
    '2-3 paragraph competitive analysis with specific numbers, ranked risk factors, '
    'a direct conclusion, and a confidence score 0-1. Never fabricate data — '
    "if a metric is missing, write 'Data not available'."
)

synthesizer_agent = Agent(
    model=SYNTHESIS_MODEL,
    system_prompt=SYNTHESIZER_SYSTEM_PROMPT,
    output_type=ResearchReport,
    retries=2,
)


async def synthesize_report(question: str, results: dict[int, str], v: ValidationResult) -> ResearchReport:
    findings = '\n\n'.join(f'=== Task {i} Result ===\n{r}' for i, r in sorted(results.items()))
    if v.warnings:
        findings += '\n\n=== Warnings ===\n' + '\n'.join(f'- {w}' for w in v.warnings)
    if v.gaps:
        findings += '\n\n=== Gaps ===\n' + '\n'.join(f'- {g}' for g in v.gaps)

    result = await synthesizer_agent.run(
        f'Research question: {question}\n\nGathered data:\n{findings}'
    )
    return result.output

## 9. The orchestrator — Plan → Execute → Validate → Synthesize

Finally, glue them together. The outer loop tries up to `MAX_REPLAN_ATTEMPTS + 1` times: if validation fails, we feed the gaps back to the planner as `additional_context` and try again. After enough attempts we proceed with whatever data we have.

In [ ]:
async def deep_search(question: str, tickers: list[str]) -> ResearchReport:
    additional_context = ''
    for attempt in range(MAX_REPLAN_ATTEMPTS + 1):
        print(f'\n[Phase 1] Planning (attempt {attempt + 1})...')
        user = question + (f'\n\nAdditional context:\n{additional_context}' if additional_context else '')
        plan = (await planner_agent.run(user)).output
        print(format_plan(plan))

        print('\n[Phase 2] Executing...')
        results = await execute_plan(plan)

        print('\n[Phase 3] Validating...')
        v = validate_research(plan, results, tickers)
        print(format_validation(v))

        if v.is_valid:
            break
        if attempt < MAX_REPLAN_ATTEMPTS:
            additional_context = 'Address the following gaps:\n' + '\n'.join(f'- {g}' for g in v.gaps + v.errors)

    print('\n[Phase 4] Synthesizing...')
    return await synthesize_report(question, results, v)

## 10. Run the full pipeline

Now run the agent end-to-end on the chip-market comparison question. With stub tools this is deterministic enough to inspect — watch the four phases in order.

In [ ]:
question = (
    "Analyze NVIDIA's financial performance and competitive position in the "
    "AI chip market. Compare with AMD and Intel on revenue growth, profitability, "
    "and risk factors. Produce a structured investment memo."
)
tickers = ['NVDA', 'AMD', 'INTC']

report = await deep_search(question, tickers)

## 11. Inspect the report

In [ ]:
print(f'Title: {report.title}\n')
print(f'Confidence: {report.confidence_score:.0%}\n')
print('Executive Summary:\n' + report.executive_summary)
print('\nConclusion:\n' + report.conclusion)

## 12. Results

You ran a full Plan → Execute → Validate → Synthesize loop, with parallel tool execution and a structured final report. **What to notice about the deep search architecture:**

* **Structured output everywhere.** Both the planner and the synthesizer return Pydantic objects,   not free-form text. This is what makes the pipeline reliable enough to put downstream code on top of.
* **Deterministic validation, not LLM-as-judge.** The validator is plain Python — it checks   ticker coverage, mathematical consistency, and dependency completion. Cheap, fast, reliable.
* **Replanning closes the loop.** When validation fails, the gaps feed back into the planner — the   agent self-corrects rather than producing a broken report.
* **Parallel execution matters.** With NVDA / AMD / INTC running concurrently, latency is roughly   the slowest task, not the sum. Chapter 7 generalises this into architectural patterns.

Swap the stub tools for the real Financial Datasets / SEC EDGAR / Tavily implementations in the chapter's `src/tools/` directory and you have a production-shaped deep research agent.